In [ ]:
import tqdm as notebook_tqdm
import torch
from rich import print
from datasets import  final_extinctionrisk_dataframe, final_extinctionrisk_noth_dataframe
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from pytorch_tabular import TabularModel
from pytorch_tabular.models import TabNetModelConfig
from pytorch_tabular.config import (
    DataConfig,
    OptimizerConfig,
    TrainerConfig,
)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device == "cuda":
    torch.set_float32_matmul_precision('high')

# MODEL 1

In [ ]:
model, data = final_extinctionrisk_dataframe()
categorical_data = data.drop(model.numeric, axis=1)
cat_col_names = list(categorical_data.columns)
num_col_names = model.numeric

print(f"Data Shape: {data.shape} | # of cat cols: {len(cat_col_names)} | # of num cols: {len(num_col_names)}")
print(f"[bold dodger_blue2] Features: {num_col_names + cat_col_names}[/bold dodger_blue2]")
print(f"[bold purple4]Target: {model.label}[/bold purple4]")

In [ ]:

train, test = train_test_split(data, random_state=42, test_size=0.2)
train, val = train_test_split(train, random_state=42, test_size=0.2)
print(f"Train Shape: {train.shape} | Val Shape: {val.shape} | Test Shape: {test.shape}")

In [ ]:
data_config = DataConfig(
    target=[
        model.label
    ],  
    continuous_cols=num_col_names,
    categorical_cols=cat_col_names,
    num_workers = 0, 
    pin_memory=True
)
trainer_config = TrainerConfig(
    batch_size=128,
    max_epochs=50,
)
optimizer_config = OptimizerConfig()

model_config = TabNetModelConfig(
    n_d = 8,
    n_a = 8,
    n_steps = 3,
    gamma = 1.3,
    n_independent = 2,
    n_shared = 2,
    virtual_batch_size=128,
    mask_type = 'sparsemax',
    task="classification",
    head = 'LinearHead',
    embedding_dropout = 0.0,
    batch_norm_continuous_input = True,
    learning_rate = 0.001,
    seed = 42,
    metrics = ["accuracy"]
)

In [ ]:
tabular_model = TabularModel(
    data_config=data_config,
    model_config=model_config,
    optimizer_config=optimizer_config,
    trainer_config=trainer_config,
    verbose=True
)

In [ ]:
tabular_model.fit(train=train, validation=val)

In [ ]:
pred_df = tabular_model.predict(test)["Extinction_risk_prediction"].to_numpy()

mapping = {'Lower_risk':2, 'Higher_risk':1}
test_y = test[model.label]
test_y = test_y.map(mapping).to_numpy()

# With Human Threats Included

In [ ]:
f1 = f1_score(test_y, pred_df, average=None)
precision = precision_score(test_y, pred_df, average=None)
recall = recall_score(test_y, pred_df, average=None)

print("F1:"+str(f1)+", Precision:"+str(precision)+", Recall:"+str(recall))

# MODEL 2

In [ ]:
model, data = final_extinctionrisk_noth_dataframe()
categorical_data = data.drop(model.numeric, axis=1)
cat_col_names = list(categorical_data.columns)
num_col_names = model.numeric

train, test = train_test_split(data, random_state=42, test_size=0.2)
train, val = train_test_split(train, random_state=42, test_size=0.2)

data_config = DataConfig(
    target=[
        model.label
    ],  
    continuous_cols=num_col_names,
    categorical_cols=cat_col_names,
    num_workers = 0, 
    pin_memory=True
)
trainer_config = TrainerConfig(
    batch_size=128,
    max_epochs=50,
)
optimizer_config = OptimizerConfig()

model_config = TabNetModelConfig(
    n_d = 8,
    n_a = 8,
    n_steps = 3,
    gamma = 1.3,
    n_independent = 2,
    n_shared = 2,
    virtual_batch_size=128,
    mask_type = 'sparsemax',
    task="classification",
    head = 'LinearHead',
    embedding_dropout = 0.0,
    batch_norm_continuous_input = True,
    learning_rate = 0.001,
    seed = 42,
    metrics = ["accuracy"]
)

tabular_model = TabularModel(
    data_config=data_config,
    model_config=model_config,
    optimizer_config=optimizer_config,
    trainer_config=trainer_config,
    verbose=True
)

tabular_model.fit(train=train, validation=val)
pred_df = tabular_model.predict(test)["Extinction_risk_prediction"].to_numpy()

mapping = {'Lower_risk':2, 'Higher_risk':1}
test_y = test[model.label]
test_y = test_y.map(mapping).to_numpy()


# Without Human Threats

In [ ]:
f1 = f1_score(test_y, pred_df, average=None)
precision = precision_score(test_y, pred_df, average=None)
recall = recall_score(test_y, pred_df, average=None)

print("F1:"+str(f1)+", Precision:"+str(precision)+", Recall:"+str(recall))